# Inspect and clean H2 data

Load `data/H2.csv`, remove duplicate and invalid bookings, select clustering-safe features, and save the cleaned dataset to `data/processed/h2_clustering.csv`. Run the code cells from top to bottom.


In [1]:

import os
import numpy as np
import pandas as pd
   # If we are inside the "notebooks" folder, go up one folder
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working folder:", os.getcwd())


Working folder: /Users/mohamedatef/ML Project/Hotel_Booking_ML-


## Step 1: load the file


In [2]:
df = pd.read_csv("data/H2.csv", skipinitialspace=True)
print("Original shape:", df.shape)



Original shape: (79330, 31)


## Step 2: clean the text


In [3]:
# Find all the text columns (every column that is not a number)
text_columns = df.select_dtypes(exclude="number").columns

# Remove the extra spaces around the text in each of those columns
for column in text_columns:
    df[column] = df[column].str.strip()

# The file writes missing values as the word "NULL", so turn them into real missing values
df = df.replace("NULL", np.nan)



## Step 3: remove duplicate rows


In [ ]:
number_of_duplicates = df.duplicated().sum()
print("Duplicate rows:", number_of_duplicates)

df = df.drop_duplicates()
df = df.reset_index(drop=True)
print("Shape after removing duplicates:", df.shape)



Duplicate rows: 25902
Shape after removing duplicates: (53428, 31)


## Step 4: remove impossible bookings


In [5]:
total_guests = df["Adults"] + df["Children"].fillna(0) + df["Babies"]
total_nights = df["StaysInWeekendNights"] + df["StaysInWeekNights"]

# A booking is bad if it has 0 guests, or 0 nights, or an extreme price
bad_rows = (total_guests == 0) | (total_nights == 0) | (df["ADR"] > 1000)
print("Bad rows removed:", bad_rows.sum())

df = df[~bad_rows]
df = df.reset_index(drop=True)



Bad rows removed: 386


## Step 5: fill the few missing Children values


In [6]:
df["Children"] = df["Children"].fillna(0)



In [7]:
# ---------- Step 5b: build date features ----------
# Join year, month and day into one text, like "2015 July 1"
text_date = df["ArrivalDateYear"].astype(str) + " " + df["ArrivalDateMonth"] + " " + df["ArrivalDateDayOfMonth"].astype(str)

# Turn that text into a real date. %Y = year, %B = full month name, %d = day
df["ArrivalDate"] = pd.to_datetime(text_date, format="%Y %B %d")

# Get the day of the week from the date (Monday, Tuesday, ...)
df["ArrivalWeekday"] = df["ArrivalDate"].dt.day_name()
# 1 if the booking waited on the waiting list, 0 if it did not
df["WasWaitlisted"] = (df["DaysInWaitingList"] > 0).astype(int)

In [8]:
# Save ALL columns (the experiments notebook reads this file)
os.makedirs("data/processed", exist_ok=True)
df.to_csv("data/processed/h2_clean_all_columns.csv", index=False)
print("Saved all columns:", df.shape)

Saved all columns: (53042, 34)


## Step 6: drop the columns we should not cluster on


In [9]:
columns_to_drop = [
    # They show the final result of the booking (leakage)
    "IsCanceled", "ReservationStatus", "ReservationStatusDate",
    # They are only known later, after the booking was made
    "AssignedRoomType", "BookingChanges", "DaysInWaitingList",
    # IDs, or too many missing / too many different values
    "Agent", "Company", "Country",
    # Not needed, because ArrivalDateMonth is enough
    "ArrivalDateYear", "ArrivalDateWeekNumber", "ArrivalDateDayOfMonth","ArrivalDate",
    # Borderline columns (we explain why in the report)
    "DepositType", "RequiredCarParkingSpaces",
]
df = df.drop(columns=columns_to_drop)



## Step 7: check and save


In [10]:
print("Final shape:", df.shape)
print("Columns kept:", list(df.columns))
print("Missing values left:", df.isna().sum().sum())

os.makedirs("data/processed", exist_ok=True)
df.to_csv("data/processed/h2_clustering.csv", index=False)
print("Saved to data/processed/h2_clustering.csv")


Final shape: (53042, 19)
Columns kept: ['LeadTime', 'ArrivalDateMonth', 'StaysInWeekendNights', 'StaysInWeekNights', 'Adults', 'Children', 'Babies', 'Meal', 'MarketSegment', 'DistributionChannel', 'IsRepeatedGuest', 'PreviousCancellations', 'PreviousBookingsNotCanceled', 'ReservedRoomType', 'CustomerType', 'ADR', 'TotalOfSpecialRequests', 'ArrivalWeekday', 'WasWaitlisted']
Missing values left: 0
Saved to data/processed/h2_clustering.csv


In [11]:
df.to_csv("data/h2_after_drop.csv", index=False)

In [12]:
print("Saved?", os.path.exists("data/processed/h2_after_drop.csv"))
print("Files in folder:", os.listdir("data/processed"))


Saved? False
Files in folder: ['h2_clustering.csv', 'h2_clean_all_columns.csv']
